# Dataset Exploration & Cleaning

This notebook performs dataset cleaning and enrichment with NACE industry codes.

**Pipeline:**
1. Load data
2. Rename columns & normalise text
3. Enrich with City / Region from Province code
4. Map ATECO → NACE with hierarchical fallback
5. Final cleaning
6. Clean dataset overview

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')
import italian_provinces

## 1. Load Data

In [2]:
BASE_DIR = os.getcwd()
file_path_csv  = os.path.join(BASE_DIR, 'dataset.csv')
file_path_xlsx = os.path.join(BASE_DIR, 'Ateco-2022-vs-NACE-Rev.-2.xlsx')

df_csv  = pd.read_csv(file_path_csv, sep=';', encoding='latin-1')
df_xlsx = pd.read_excel(file_path_xlsx, sheet_name='NACE Rev. 2 vs Ateco 2022')

print(f'CSV  -> {df_csv.shape[0]:,} rows, {df_csv.shape[1]} columns')
print(f'XLSX -> {df_xlsx.shape[0]:,} rows, {df_xlsx.shape[1]} columns')

CSV  -> 20,436 rows, 42 columns
XLSX -> 3,157 rows, 4 columns


## 2. Rename Columns & Normalise Text

In [3]:
# Translate Italian column names to English
df_csv.rename(columns={
    'Natura':     'Nature',
    'Fatturato':  'Revenue',
    'Dipendenti': 'Employees',
    'Provincia':  'Province'
}, inplace=True)

# Normalise ATECO descriptions: lowercase, strip punctuation
df_csv['ATECO_Desc'] = (
    df_csv['ATECO_Desc']
    .str.lower()
    .str.replace(r'[^\w\s]', '', regex=True)
    .str.strip()
)

df_csv.head(3)

,ATECO,ATECO_Desc,Nature,Revenue,Employees,Province,M-35,M-34,M-33,M-32,...,M-9,M-8,M-7,M-6,M-5,M-4,M-3,M-2,M-1,M-0
0,G.47.71.00,commercio al dettaglio di articoli di abbiglia...,1.a Impresa,93840,2,PA,80,718,3888,6255,...,65919,58362,69779,62338,71332,54796,91169,104473,127672,139105
1,H.49.41.00,trasporto di merci su strada,1.a Impresa,1242280,4,MB,44520,43793,54545,52400,...,78015,73822,83936,68864,107911,37607,100365,105866,84089,95330
2,G.47.91.30,commercio al dettaglio di qualsiasi tipo di pr...,1.a Impresa,44451177,159,MI,65703,87812,65768,59904,...,35367,41628,60515,57786,85252,76067,86415,75573,101941,237050


## 3. Geographic Enrichment: Province -> City & Region

In [4]:
# Look up City and Region for every non-null Province code
df_csv.loc[df_csv['Province'].notna(), ['City', 'Region']] = (
    df_csv.loc[df_csv['Province'].notna(), 'Province']
    .apply(lambda p: pd.Series(
        italian_provinces.italian_provinces.get(p.strip().upper(), {'City': None, 'Region': None})
    ))
)

# Place City and Region immediately after Province for readability
cols = list(df_csv.columns)
for col in ['City', 'Region']:
    if col in cols:
        cols.remove(col)
province_idx = cols.index('Province')
cols = cols[:province_idx+1] + ['City', 'Region'] + cols[province_idx+1:]
df_csv = df_csv[cols]

# Diagnostic: flag any Province codes that could not be mapped
missing_geo = df_csv[df_csv['Province'].notna() & (df_csv['City'].isna() | df_csv['Region'].isna())]
print(f'Provinces with no City/Region mapping: {len(missing_geo)}')
if len(missing_geo):
    print(missing_geo[['Province', 'City', 'Region']])

Provinces with no City/Region mapping: 1
      Province City Region
13821       EE  NaN    NaN


## 4. ATECO -> NACE Mapping with Hierarchical Fallback

ATECO codes in the dataset follow the format `X.xx.xx.xx` (e.g. `G.47.71.00`).  
After stripping the sector prefix (e.g. `G.`) we obtain `47.71.00`, which may not exist
verbatim in the XLSX reference file that uses ATECO 2007 nomenclature.

**Multi-level matching strategy:**

| Priority | Type | Example |
|---|---|---|
| 1 | Exact match | `47.71.00` found directly |
| 2 | Truncate to 7 chars | `47.71.0` |
| 3 | Truncate to 6 chars | `47.71.` stripped to `47.71` |
| 4 | Truncate to 5 chars | `47.71` |
| 5 | Truncate to 4 chars | `47.7` |
| 6 | Truncate to 2 chars | `47` (macro-sector fallback) |

In [5]:
# Assign column names and normalise descriptions in the reference file
df_xlsx.columns = ['NACE', 'NACE_Desc', 'ATECO_2007', 'ATECO_2007_Desc']

df_xlsx['ATECO_2007_Desc'] = (
    df_xlsx['ATECO_2007_Desc']
    .str.lower()
    .str.replace(r'[^\w\s]', '', regex=True)
    .str.strip()
)

# Build the lookup dict: ATECO_2007 code -> {NACE, NACE_Desc}
xl_clean = df_xlsx.dropna(subset=['NACE', 'ATECO_2007'])
ateco_to_nace: dict = xl_clean.set_index('ATECO_2007')[['NACE', 'NACE_Desc']].to_dict('index')

print(f'ATECO->NACE lookup built: {len(ateco_to_nace):,} entries')

ATECO->NACE lookup built: 3,157 entries


In [6]:
def find_nace(ateco_norm: str) -> tuple[str | None, str | None, str]:
    """
    Map a normalised ATECO code (e.g. '47.71.00') to its NACE Rev. 2
    equivalent using a hierarchical fallback strategy.

    Returns:
        (nace_code, nace_desc, match_type)
        match_type is one of:
          - 'exact'       : verbatim match in the reference table
          - 'prefix_N'    : matched after truncating to N characters
          - 'missing_ateco': input was null or empty
          - 'no_match'    : no match found at any level
    """
    if pd.isna(ateco_norm) or str(ateco_norm).strip() == '':
        return None, None, 'missing_ateco'

    code = str(ateco_norm).strip()

    # Priority 1: exact match
    if code in ateco_to_nace:
        r = ateco_to_nace[code]
        return r['NACE'], r['NACE_Desc'], 'exact'

    # Priority 2-6: progressively truncate the code (7 -> 6 -> 5 -> 4 -> 2 chars)
    for length in [7, 6, 5, 4, 2]:
        prefix = code[:length].rstrip('.')   # strip any trailing dot left by truncation
        if prefix and prefix in ateco_to_nace:
            r = ateco_to_nace[prefix]
            return r['NACE'], r['NACE_Desc'], f'prefix_{length}'

    return None, None, 'no_match'

In [7]:
# Strip the sector letter prefix (e.g. 'G.') to obtain a plain numeric ATECO code
df_csv['ATECO_norm'] = df_csv['ATECO'].str[2:]

# Apply the mapping to every row
nace_results = df_csv['ATECO_norm'].apply(
    lambda x: pd.Series(find_nace(x), index=['NACE', 'NACE_Desc', 'nace_match_type'])
)
df_csv = pd.concat([df_csv, nace_results], axis=1)

# Matching quality report
print('=== ATECO -> NACE mapping quality ===')
match_stats = df_csv['nace_match_type'].value_counts()
for mtype, count in match_stats.items():
    pct = 100 * count / len(df_csv)
    print(f'  {mtype:<20} {count:>6,}  ({pct:.1f}%)')

total_with_ateco = df_csv['ATECO'].notna().sum()
total_with_nace  = df_csv[df_csv['ATECO'].notna() & df_csv['NACE'].notna()].shape[0]
print(f'\nRows with ATECO:  {total_with_ateco:,}')
print(f'NACE assigned:    {total_with_nace:,} ({100*total_with_nace/total_with_ateco:.1f}%)')

=== ATECO -> NACE mapping quality ===
  exact                18,106  (88.6%)
  prefix_2                567  (2.8%)
  prefix_4                541  (2.6%)
  prefix_6                497  (2.4%)
  missing_ateco           403  (2.0%)
  prefix_7                322  (1.6%)

Rows with ATECO:  20,049
NACE assigned:    20,033 (99.9%)


In [8]:
# Inspect rows where ATECO is present but no NACE could be assigned
unmatched = df_csv[df_csv['nace_match_type'] == 'no_match']
print(f'Rows without NACE despite having an ATECO code: {len(unmatched)}')
if len(unmatched):
    print(unmatched[['ATECO', 'ATECO_Desc']].drop_duplicates())

Rows without NACE despite having an ATECO code: 0


## 5. Final Cleaning

In [9]:
# Drop rows missing any key column
df_clean = df_csv.dropna(subset=['ATECO', 'Province', 'City', 'Region', 'NACE']).copy()

# Drop rows where all monthly M-xx columns are zero (no activity recorded)
m_cols = [f'M-{i}' for i in range(36)]
df_clean = df_clean[df_clean[m_cols].sum(axis=1) != 0]

# Remove intermediate / source columns no longer needed in the final dataset
cols_to_drop = ['ATECO', 'ATECO_Desc', 'ATECO_norm', 'Nature', 'nace_match_type']
df_clean = df_clean.drop(columns=[c for c in cols_to_drop if c in df_clean.columns])

# Move NACE and NACE_Desc to the front for convenience
front_cols = ['NACE', 'NACE_Desc']
other_cols = [c for c in df_clean.columns if c not in front_cols]
df_clean = df_clean[front_cols + other_cols]

print(f'Clean dataset: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns')
df_clean.head()

Clean dataset: 16,791 rows x 43 columns


,NACE,NACE_Desc,Revenue,Employees,Province,City,Region,M-35,M-34,M-33,...,M-9,M-8,M-7,M-6,M-5,M-4,M-3,M-2,M-1,M-0
0,47.71,Retail sale of clothing in specialised stores,93840,2,PA,Palermo,Sicilia,80,718,3888,...,65919,58362,69779,62338,71332,54796,91169,104473,127672,139105
1,49.41,Freight transport by road,1242280,4,MB,Monza,Lombardia,44520,43793,54545,...,78015,73822,83936,68864,107911,37607,100365,105866,84089,95330
2,47.91,Retail sale via mail order houses or via Internet,44451177,159,MI,Milano,Lombardia,65703,87812,65768,...,35367,41628,60515,57786,85252,76067,86415,75573,101941,237050
3,77.40,Leasing of intellectual property and similar p...,49918687,53,RM,Roma,Lazio,0,0,0,...,63667,74293,82637,80798,105440,74817,73151,76448,88819,85649
5,62.01,Computer programming activities,7115458,18,MI,Milano,Lombardia,47372,38998,32265,...,62883,58357,66783,64001,96385,57581,58713,72002,76832,76719


## 6. Clean Dataset Overview

In [10]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 16791 entries, 0 to 20435
Data columns (total 43 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   NACE       16791 non-null  str  
 1   NACE_Desc  16791 non-null  str  
 2   Revenue    16791 non-null  int64
 3   Employees  16791 non-null  int64
 4   Province   16791 non-null  str  
 5   City       16791 non-null  str  
 6   Region     16791 non-null  str  
 7   M-35       16791 non-null  int64
 8   M-34       16791 non-null  int64
 9   M-33       16791 non-null  int64
 10  M-32       16791 non-null  int64
 11  M-31       16791 non-null  int64
 12  M-30       16791 non-null  int64
 13  M-29       16791 non-null  int64
 14  M-28       16791 non-null  int64
 15  M-27       16791 non-null  int64
 16  M-26       16791 non-null  int64
 17  M-25       16791 non-null  int64
 18  M-24       16791 non-null  int64
 19  M-23       16791 non-null  int64
 20  M-22       16791 non-null  int64
 21  M-21       16791 non-null  i

In [11]:
null_pct = df_clean.isnull().mean() * 100
cols_with_nulls = null_pct[null_pct > 0]
if cols_with_nulls.empty:
    print('No null values remaining in the clean dataset.')
else:
    print('Columns with null values:')
    print(cols_with_nulls)

No null values remaining in the clean dataset.


In [12]:
df_clean.describe()

,Revenue,Employees,M-35,M-34,M-33,M-32,M-31,M-30,M-29,M-28,...,M-9,M-8,M-7,M-6,M-5,M-4,M-3,M-2,M-1,M-0
count,1.679100e+04,16791.000000,1.679100e+04,1.679100e+04,1.679100e+04,16791.000000,16791.000000,16791.000000,16791.000000,16791.000000,...,16791.000000,16791.000000,16791.000000,16791.000000,16791.000000,16791.000000,16791.000000,16791.000000,16791.000000,16791.000000
mean,4.330619e+06,24.095468,5.088587e+02,5.054312e+02,5.526819e+02,446.813174,495.822762,430.419630,411.871777,320.021083,...,438.674528,436.742481,480.899946,424.438926,469.016378,298.429635,450.787743,500.544875,511.531535,556.302245
std,8.467519e+06,175.056498,8.674717e+03,1.035797e+04,9.365590e+03,4566.415205,5894.118955,2753.181369,2463.722744,2099.951327,...,2784.890128,2911.332342,3159.152011,2720.646471,3169.554300,2297.950571,3096.513599,3332.131041,3530.061692,4246.125080
min,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,-98444.000000,-192.000000,-117.000000,-115.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-343.000000,0.000000,0.000000
25%,9.122500e+03,1.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,7.638460e+05,6.000000,1.100000e+01,1.200000e+01,1.400000e+01,10.000000,6.000000,5.000000,8.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,3.990404e+06,18.000000,1.350000e+02,1.450000e+02,1.590000e+02,137.000000,143.000000,126.000000,127.500000,75.000000,...,93.000000,88.000000,100.000000,85.000000,91.000000,38.000000,82.000000,99.000000,87.000000,76.000000
max,5.792155e+07,19555.000000,1.069153e+06,1.301728e+06,1.159983e+06,508106.000000,666519.000000,124945.000000,113392.000000,87150.000000,...,80319.000000,121988.000000,134135.000000,80798.000000,107911.000000,94090.000000,139446.000000,117746.000000,135577.000000,237050.000000


In [13]:
print('=== Top 15 NACE sectors by number of records ===')
(
    df_clean.groupby(['NACE', 'NACE_Desc'])
    .size()
    .sort_values(ascending=False)
    .head(15)
    .rename('count')
    .to_frame()
)

=== Top 15 NACE sectors by number of records ===


,,count
NACE,NACE_Desc,
47.91,Retail sale via mail order houses or via Internet,1107
47.71,Retail sale of clothing in specialised stores,389
53.20,Other postal and courier activities,309
84,Public administration and defence; compulsory social security,304
46.49,Wholesale of other household goods,281
82.99,Other business support service activities n.e.c.,221
46.46,Wholesale of pharmaceutical goods,214
46.69,Wholesale of other machinery and equipment,213
47.78,Other retail sale of new goods in specialised stores,210


In [14]:
print('=== Record distribution by Region ===')
(
    df_clean['Region']
    .value_counts()
    .rename('count')
    .to_frame()
)

=== Record distribution by Region ===


,count
Region,
Lombardia,3375
Toscana,2095
Emilia-Romagna,2093
Lazio,1963
Veneto,1624
Piemonte,1243
Sicilia,841
Campania,569
Marche,510


## 7. Export the dataset

In [15]:
# Export the enriched dataset directly to CSV
output_path = os.path.join(BASE_DIR, 'dataset_eda.csv')
df_clean.to_csv(output_path, index=False, sep=',', encoding='latin-1')
print(f'Dataset exported: {output_path}')
print(f'Shape: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns')

Dataset exported: /Users/sara/Desktop/Master RBS/10 Capstone/data/dataset_eda.csv
Shape: 16,791 rows x 43 columns
